In [1]:
import os
import json
import torch
import pandas as pd
import logging
import importlib
from pathlib import Path
import numpy as np
import random
import matplotlib.pyplot as plt

import sys
PROJECT_DIR = "/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER"
SRC_DIR = str(Path(PROJECT_DIR) / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import multiomic_transformer.utils.data_formatter as data_formatter
import multiomic_transformer.utils.experiment_handler as experiment_handler

random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

# Training Data Preparation and Caching

The model uses cached per-chromosome data to help speed up training. Here, we will go through the process of building the necessary tensors and reference files to train the gene prediction model.

## Set up the TrainingDataFormatter

The training data formatter ensures that the correct data files and output directories exist and helps to keep everything standardized.

In [2]:
importlib.reload(data_formatter)

<module 'multiomic_transformer.utils.data_formatter' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/src/multiomic_transformer/utils/data_formatter.py'>

In [13]:
# Path to the project directory (same as Git repository root)
project_dir = Path(PROJECT_DIR)

# Path to the training output directory. Used to store the preprocessing config
output_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments")

# Name of the dataset / experiment to run
experiment_name = "Macrophage_buffer_2_raw_muon_preprocessing"

# Organism code for the dataset. Supports either "mm10" or "hg38"
organism_code = "hg38"

# List of samples in the training datset. 
# Each of these should have its own subdirectory in the processed data directory
sample_names = ["buffer_2"]

# List of chromosomes. Used to split the data by chromsome for caching and training.
# Should be in the format "chr1", "chr2", etc. and should match the chromosome names in the processed data files.
chrom_list = [f"chr{i}" for i in range(1, 21)]

tdf = data_formatter.TrainingDataFormatter(
    project_dir=project_dir,
    experiment_name=experiment_name,
    organism_code=organism_code,
    sample_names=sample_names,
    chrom_list=chrom_list,
    output_dir=output_dir / experiment_name,
)

with open(tdf.output_dir / "file_paths.json", "r") as f:
    file_paths = json.load(f)

Loaded existing settings from /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/processed/Macrophage_buffer_2_raw_muon_preprocessing/settings.json


## Sample-Level Data Preparation

We first load the pseudobulk data from the Muon QC and preprocessing pipeline. The data files are loaded and the peak and gene names are standardized.

We load the pseudobulk data for the sample, which also creates a list of gene names. The `split_genes_into_tfs_and_tgs` function checks which of the genes in the gene name list overlap with an organism-specific [file of TF names from CIS-BP](https://mitra.stanford.edu/kundaje/marinovg/oak/various/motifs/CIS-BP/TF_Information_all_motifs.txt) to create `tfs` and `tgs` attributes.


In [6]:
sample_name = sample_names[0]

pseudobulk_rna_df = tdf.load_pseudobulk_rna_df(sample_name)

logging.info(f"Number of genes: {len(tdf.genes):,}")

# Next the genes are classified as TFs or TGs by checking which genes in the data are in the reference TF list. 
tdf.tfs, tdf.tgs = tdf.split_genes_into_tfs_and_tgs(tdf.genes)
logging.info(f"  - TFs: {len(tdf.tfs):,} (First 3: {tdf.tfs[:3]})")
logging.info(f"  - TGs: {len(tdf.tgs):,} (First 3: {tdf.tgs[:3]})")

Number of genes: 18,862
  - TFs: 1,223 (First 3: ['2010315B03RIK', '2310033P09RIK', '2610008E11RIK'])
  - TGs: 17,627 (First 3: ['0610005C13RIK', '0610009E02RIK', '0610009L18RIK'])


The pseudobulk ATAC data is loaded, which also creates a list of peak names

In [7]:
pseudobulk_atac_df = tdf.load_pseudobulk_atac_df(sample_name)

logging.info(f"Number of peaks: {len(tdf.peaks):,} (First 3: {tdf.peaks[:3]})")

Number of peaks: 199,885 (First 3: ['chr1:3035460-3036350', 'chr1:3044677-3045614', 'chr1:3062482-3063387'])


A BED file of the peak locations is created using `create_peak_bed_file` and the peak locations are stored as a dataframe in `tdf.peak_locs_df`.

In [7]:
tdf.create_peak_bed_file(pseudobulk_atac_df, sample_name)
display(tdf.peak_locs_df.head())

,chrom,start,end,strand,peak_id
0,chr1,180546,181326,.,chr1:180546-181326
1,chr1,191323,192133,.,chr1:191323-192133
2,chr1,267079,267655,.,chr1:267079-267655
3,chr1,267759,268290,.,chr1:267759-268290
4,chr1,270895,271779,.,chr1:270895-271779


### Calculate peak to gene distance

Next, we will use the peak bed file created when loading the peak pseudobulk dataset to calculate the distance between each peak and each gene within 1 Mb. This will create a bias tensor to help the model prioritize genes that are closer to peaks when training the peak-TG cross-attention portion of the model.

In [8]:
tdf.create_peak_to_tg_distance_file(
    sample_name=sample_name,
    force_recalculate=False,
    filter_to_nearest_gene=False,
)

  - Loading saved peak to TSS distance file


,peak_chr,peak_start,peak_end,peak_id,gene_chr,gene_start,gene_end,target_id,TSS_dist,TSS_dist_score
462657,chr19,18385677,18386562,chr19:18385677-18386562,chr19,18386562,18386563,MIR3189,0,1.000000
510025,chr22,20298984,20299760,chr22:20298984-20299760,chr22,20299760,20299761,PRODHLP,0,1.000000
116917,chr3,179649316,179650199,chr3:179649316-179650199,chr3,179650200,179650201,H3P13,1,0.999950
67487,chr2,74548651,74549458,chr2:74548651-74549458,chr2,74549459,74549460,LOXL3,1,0.999950
141596,chr5,59034128,59035026,chr5:59034128-59035026,chr5,59035027,59035028,PDE4D-AS1,1,0.999950
...,...,...,...,...,...,...,...,...,...,...
270555,chr10,132684039,132684794,chr10:132684039-132684794,chr10,132784765,132784766,NKX6-2,99971,0.006748
25224,chr1,108063441,108064341,chr1:108063441-108064341,chr1,107964367,107964368,VAV3-AS1,99974,0.006747
443902,chr18,55328041,55328931,chr18:55328041-55328931,chr18,55228955,55228956,TCF4,99976,0.006746
182363,chr6,109877341,109878271,chr6:109877341-109878271,chr6,109978256,109978257,GPR6,99985,0.006743


## Chromosome-Specific Data Formatting


In [9]:
total_TG_pseudobulk_global, pseudobulk_chrom_dict = tdf.aggregate_pseudobulk_datasets(force_recalculate=False)
total_TG_pseudobulk_global.head()

  - Found existing global and per-chrom pseudobulk; loading...


,AAACAGCCATAGGCGA-1,AAACCAACAATCCTAG-1,AAACCGCGTGCTCACC-1,AAAGCACCATAATGTC-1,AAAGCACCATTAAGTC-1,AAAGCCCGTGGAAACG-1,AAATCCGGTTGCACAA-1,AAATGCCTCGAGGAAC-1,AAATGGCCACAAACTT-1,AACAAGCCAGTCTAGC-1,...,TTTATGGAGCTTGCTC-1,TTTATGGAGGATGATG-1,TTTCACCCATGATTGT-1,TTTCAGTTCCCTGTTA-1,TTTCCACCACCATATG-1,TTTCCGGGTAGTTAAC-1,TTTCCGGGTTGCAGTA-1,TTTCTCACAGGCTTCG-1,TTTGACCGTGAAACAA-1,TTTGGTGCAGAATGAC-1
A1BG,0.145938,0.375733,0.031888,0.075666,0.019815,0.267360,0.081609,-0.068398,0.136295,0.174090,...,0.093855,-0.062284,-0.071432,-0.176623,0.142207,-0.055086,0.161324,0.077176,0.053658,-0.062983
A2M,0.027574,-0.130627,-0.063552,0.030216,0.100016,0.071931,0.341915,-0.120708,0.139898,-0.101431,...,-0.134470,0.426040,-0.233120,-0.300487,0.268734,0.191113,-0.396636,0.318500,0.335033,0.329044
A2ML1,0.151865,0.097345,-0.099689,0.902793,0.411699,-0.059665,-0.086548,-0.087312,-0.064451,-0.098521,...,-0.057896,-0.099689,-0.044337,-0.030960,-0.057973,0.029453,-0.099689,-0.083345,-0.097404,-0.062819
A3GALT2,-0.155416,-0.089188,-0.080371,0.077545,0.357054,-0.121030,-0.021690,-0.144277,-0.134403,0.045692,...,-0.124641,-0.132327,-0.123117,0.344077,0.089098,0.070931,-0.164678,0.054090,0.062294,-0.044594
A4GALT,-0.121337,0.134273,-0.012551,-0.128767,-0.128767,0.049067,-0.055950,-0.093198,-0.089228,0.252164,...,-0.111661,-0.128767,0.029071,-0.116722,-0.114493,-0.127252,-0.066699,-0.120236,-0.112274,-0.073179


In [10]:
tdf.create_chrom_files(total_TG_pseudobulk_global, pseudobulk_chrom_dict, force_recalculate=False, verbose=False)

  - All required global files exist. Skipping preprocessing.
  - Number of chromosomes: 20: ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20']
  - Processing chromosomes for dataset: Macrophage_buffer_1_raw_muon_preprocessing_new_cache
  - All required output files exist for chr1. Skipping preprocessing.
  - All required output files exist for chr2. Skipping preprocessing.
  - All required output files exist for chr3. Skipping preprocessing.
  - All required output files exist for chr4. Skipping preprocessing.
  - All required output files exist for chr5. Skipping preprocessing.
  - All required output files exist for chr6. Skipping preprocessing.
  - All required output files exist for chr7. Skipping preprocessing.
  - All required output files exist for chr8. Skipping preprocessing.
  - All required output files exist for chr9. Skipping preprocessing.
  - All requi

We can check that all of the required cached files exist for the experiment.

In [ ]:
num_missing_files, missing_file_dict = tdf.check_cached_file_exist()
logging.info(f"Missing {num_missing_files} cached files")
missing_file_dict

All required files are present.


Missing 0 cached files


{'sample': [], 'chromosome': {}}

In [ ]:
tdf.save_settings()